# ex1_chedi

In [1]:
"""
Pathwise Sensitivity for Down-and-Out Barrier Option
Brownian Bridge barrier crossing correction (Glasserman §6.4)
"""
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import norm

# ── Parameters ────────────────────────────────────────────────────────────────
r, sigma, T = 0.05, 0.5, 1.0
s0, K, B    = 100.0, 110.0, 90.0
N  = 200_000   # paths
n  = 52        # weekly steps
dt = T / n
np.random.seed(42)

# ── Analytical closed form ─────────────────────────────────────────────────────
def do_barrier_call(s, K, B, r, sigma, T):
    lam = (r + 0.5*sigma**2) / sigma**2
    sqT = sigma * np.sqrt(T)
    x1  = np.log(s/K)  / sqT + lam*sqT
    y1  = np.log(B**2/(s*K)) / sqT + lam*sqT
    C   = (s  * norm.cdf(x1)
          - K * np.exp(-r*T) * norm.cdf(x1 - sqT)
          - s  * (B/s)**(2*lam)     * norm.cdf(y1)
          + K * np.exp(-r*T) * (B/s)**(2*lam-2) * norm.cdf(y1 - sqT))
    return C

V_analytic = do_barrier_call(s0, K, B, r, sigma, T)

# ── Core simulation (batched to save memory) ───────────────────────────────────
def simulate_batch(N, n, s0, r, sigma, T, B, K, seed=None):
    if seed is not None:
        np.random.seed(seed)
    dt      = T/n
    sqrt_dt = np.sqrt(dt)
    Z       = np.random.standard_normal((N, n))          # (N,n)
    log_inc = (r - 0.5*sigma**2)*dt + sigma*sqrt_dt*Z   # (N,n)
    log_S   = np.log(s0) + np.cumsum(log_inc, axis=1)   # (N,n)
    S       = np.exp(log_S)                               # (N,n)

    # S_prev: prepend s0 before first step
    Sp = np.empty((N, n), dtype=np.float64)
    Sp[:, 0]  = s0
    Sp[:, 1:] = S[:, :-1]

    alive = np.all(S > B, axis=1)                        # (N,)

    # Bridge crossing probabilities
    lp = np.log(np.maximum(Sp, B+1e-12) / B)
    ln = np.log(np.maximum(S,  B+1e-12) / B)
    ba = (Sp > B) & (S > B)                              # both above barrier
    pc = np.where(ba, np.exp(-2.0*lp*ln/(sigma**2*dt)), 0.0)  # (N,n)

    log_surv = np.sum(np.log(np.maximum(1.0 - pc, 1e-300)), axis=1)
    sw       = np.where(alive, np.exp(log_surv), 0.0)

    ST    = S[:, -1]
    payoff = np.maximum(ST - K, 0.0) * np.exp(-r*T)
    f     = sw * payoff   # weighted payoff

    # ── Pathwise Delta ────────────────────────────────────────────────────────
    # dp_cross/ds0 = pc * (-2/(sigma^2*dt*s0)) * (ln + lp)
    dpc_ds0   = np.where(ba, pc * (-2.0/(sigma**2*dt*s0)) * (ln + lp), 0.0)
    dlsw_ds0  = np.sum(-dpc_ds0 / np.maximum(1.0 - pc, 1e-300), axis=1)
    dpay_ds0  = np.exp(-r*T) * (ST > K).astype(float) * (ST / s0)
    delta_raw = np.where(alive, sw*payoff*dlsw_ds0 + sw*dpay_ds0, 0.0)

    # ── Pathwise Vega ─────────────────────────────────────────────────────────
    # d(log S_t)/dsigma = sum_{j<=t} (Z_j*sqrt(dt) - sigma*dt)
    isig  = Z*sqrt_dt - sigma*dt              # (N,n)
    csig  = np.cumsum(isig, axis=1)           # cumulative to step t
    csig0 = np.empty_like(csig)
    csig0[:, 0]  = 0.0
    csig0[:, 1:] = csig[:, :-1]              # cumulative to step t-1

    dpc_dsig = np.where(ba,
        pc * (4.0*lp*ln/(sigma**3*dt)
              + (-2.0/(sigma**2*dt)) * (csig0*ln + lp*csig)),
        0.0)
    dlsw_dsig  = np.sum(-dpc_dsig / np.maximum(1.0-pc, 1e-300), axis=1)
    dpay_dsig  = np.exp(-r*T) * (ST>K).astype(float) * (ST * csig[:, -1])
    vega_raw   = np.where(alive, sw*payoff*dlsw_dsig + sw*dpay_dsig, 0.0)

    return f, delta_raw, vega_raw, Z, S, alive

f, delta_raw, vega_raw, Z_all, S_all, alive_all = simulate_batch(N, n, s0, r, sigma, T, B, K, seed=42)

V_mc     = np.mean(f);          V_se     = np.std(f)/np.sqrt(N)
Delta_mc = np.mean(delta_raw);  Delta_se = np.std(delta_raw)/np.sqrt(N)
Vega_mc  = np.mean(vega_raw);   Vega_se  = np.std(vega_raw)/np.sqrt(N)

# ── Finite-difference benchmarks ──────────────────────────────────────────────
hs, hv = 0.1, 0.001
Delta_fd = (do_barrier_call(s0+hs,K,B,r,sigma,T) - do_barrier_call(s0-hs,K,B,r,sigma,T)) / (2*hs)
Vega_fd  = (do_barrier_call(s0,K,B,r,sigma+hv,T) - do_barrier_call(s0,K,B,r,sigma-hv,T)) / (2*hv)

print("="*55)
print(f"  {'Quantity':<20} {'MC Estimate':>12}  {'Analytic/FD':>12}")
print("-"*55)
print(f"  {'Price V':<20} {V_mc:>12.6f}  {V_analytic:>12.6f}")
print(f"  {'Delta':<20} {Delta_mc:>12.6f}  {Delta_fd:>12.6f}")
print(f"  {'Vega':<20} {Vega_mc:>12.6f}  {Vega_fd:>12.6f}")
print("="*55)

# ── Convergence study ─────────────────────────────────────────────────────────
Ns = [2000, 5000, 10000, 50000, 100000, 200000]
res = {'price': [], 'delta': [], 'vega': [],
       'pse':   [], 'dse':   [], 'vse':  []}
for Ni in Ns:
    fi, di, vi, *_ = simulate_batch(Ni, n, s0, r, sigma, T, B, K, seed=1)
    res['price'].append(np.mean(fi));  res['pse'].append(np.std(fi)/np.sqrt(Ni))
    res['delta'].append(np.mean(di));  res['dse'].append(np.std(di)/np.sqrt(Ni))
    res['vega'].append(np.mean(vi));   res['vse'].append(np.std(vi)/np.sqrt(Ni))

# ── Plots ─────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# Panel 1 – sample paths
ax1 = fig.add_subplot(gs[0, 0])
t_arr = np.linspace(0, T, n+1)
np.random.seed(7)
ns = 40
Zs   = np.random.standard_normal((ns, n))
logS = np.log(s0) + np.cumsum((r-0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Zs, axis=1)
Ss   = np.exp(logS)
Sfull = np.hstack([np.full((ns,1), s0), Ss])
for i in range(ns):
    col = 'steelblue' if np.all(Ss[i]>B) else 'tomato'
    ax1.plot(t_arr, Sfull[i], lw=0.5, alpha=0.55, color=col)
ax1.axhline(B, color='red',    lw=1.8, ls='--', label=f'Barrier B={B}')
ax1.axhline(K, color='purple', lw=1.5, ls=':',  label=f'Strike K={K}')
ax1.set_xlabel('Time t'); ax1.set_ylabel('S(t)')
ax1.set_title('Sample GBM Paths\n(blue=survived, red=knocked out)')
ax1.legend(fontsize=8)

# Panel 2 – Price convergence
ax2 = fig.add_subplot(gs[0, 1])
ax2.errorbar(Ns, res['price'], yerr=[1.96*s for s in res['pse']],
             fmt='o-', color='steelblue', capsize=4, label='MC Price')
ax2.axhline(V_analytic, color='red', ls='--', lw=1.5, label=f'Exact={V_analytic:.4f}')
ax2.set_xscale('log'); ax2.set_xlabel('N'); ax2.set_ylabel('Price')
ax2.set_title('Option Price Convergence'); ax2.legend(fontsize=8)

# Panel 3 – Delta convergence
ax3 = fig.add_subplot(gs[0, 2])
ax3.errorbar(Ns, res['delta'], yerr=[1.96*s for s in res['dse']],
             fmt='s-', color='darkorange', capsize=4, label='Pathwise Δ')
ax3.axhline(Delta_fd, color='red', ls='--', lw=1.5, label=f'FD Δ={Delta_fd:.4f}')
ax3.set_xscale('log'); ax3.set_xlabel('N'); ax3.set_ylabel('Delta')
ax3.set_title('Delta Convergence (Pathwise)'); ax3.legend(fontsize=8)

# Panel 4 – Vega convergence
ax4 = fig.add_subplot(gs[1, 0])
ax4.errorbar(Ns, res['vega'], yerr=[1.96*s for s in res['vse']],
             fmt='^-', color='green', capsize=4, label='Pathwise ν')
ax4.axhline(Vega_fd, color='red', ls='--', lw=1.5, label=f'FD ν={Vega_fd:.4f}')
ax4.set_xscale('log'); ax4.set_xlabel('N'); ax4.set_ylabel('Vega')
ax4.set_title('Vega Convergence (Pathwise)'); ax4.legend(fontsize=8)

# Panel 5 – Delta distribution
ax5 = fig.add_subplot(gs[1, 1])
lo5, hi5 = np.percentile(delta_raw, 1), np.percentile(delta_raw, 99)
d_trim = delta_raw[(delta_raw>lo5) & (delta_raw<hi5)]
ax5.hist(d_trim, bins=80, color='darkorange', alpha=0.7, density=True)
ax5.axvline(Delta_mc, color='darkred', ls='--', lw=2, label=f'Δ={Delta_mc:.4f}')
ax5.axvline(Delta_fd, color='black',   ls=':',  lw=2, label=f'FD={Delta_fd:.4f}')
ax5.set_xlabel('Pathwise Δ contribution'); ax5.set_ylabel('Density')
ax5.set_title('Pathwise Delta Distribution\n(middle 98%)'); ax5.legend(fontsize=8)

# Panel 6 – Vega distribution
ax6 = fig.add_subplot(gs[1, 2])
lo6, hi6 = np.percentile(vega_raw, 1), np.percentile(vega_raw, 99)
v_trim = vega_raw[(vega_raw>lo6) & (vega_raw<hi6)]
ax6.hist(v_trim, bins=80, color='green', alpha=0.7, density=True)
ax6.axvline(Vega_mc, color='darkred', ls='--', lw=2, label=f'ν={Vega_mc:.4f}')
ax6.axvline(Vega_fd, color='black',   ls=':',  lw=2, label=f'FD={Vega_fd:.4f}')
ax6.set_xlabel('Pathwise ν contribution'); ax6.set_ylabel('Density')
ax6.set_title('Pathwise Vega Distribution\n(middle 98%)'); ax6.legend(fontsize=8)

fig.suptitle(
    'Down-and-Out Barrier Call: Pathwise Sensitivity with Brownian Bridge Correction\n'
    f'r={r}, σ={sigma}, T={T}, s₀={s0}, K={K}, B={B}  |  N={N:,}, n={n} steps',
    fontsize=12, fontweight='bold')
plt.savefig('barrier_option_pathwise.png', dpi=150, bbox_inches='tight')
plt.close()

# To this:
plt.savefig('barrier_option_pathwise.png', dpi=150, bbox_inches='tight')
plt.show()  # ← add this line
plt.close()
print("Plot saved.")

  Quantity              MC Estimate   Analytic/FD
-------------------------------------------------------
  Price V                  8.548165      8.595814
  Delta                    0.838267      0.849358
  Vega                     4.700012      4.550680
Plot saved.


/var/folders/th/b_cdn0h107lcl7b1cp9xd7zc0000gn/T/ipykernel_14983/3464878370.py:194: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()  # ← add this line


In [2]:
"""
mc_barrier_pathwise.py

Monte Carlo estimator for a down-and-out barrier call under GBM.
Computes price, Delta (dV/dS0) and Vega (dV/dsigma) using the pathwise method
and Brownian-bridge barrier crossing correction (Glasserman Sec. 6.4).

Author: ChatGPT (seeded for reproducibility)
"""

import numpy as np
import math

# ---------------------------
# Core simulation function
# ---------------------------
def mc_barrier_pathwise(N_paths=100000, N_steps=100, S0=100.0, K=110.0, B=90.0, r=0.05, sigma=0.5, T=1.0, seed=12345):
    """
    Monte Carlo with Brownian-bridge barrier correction and pathwise sensitivities.
    Returns dict with price, delta, vega and standard errors.
    """
    np.random.seed(seed)
    dt = T / N_steps
    sqrt_dt = math.sqrt(dt)
    lnB = math.log(B)
    lnS0 = math.log(S0)

    # simulate standard normals: shape (N_paths, N_steps)
    Z = np.random.normal(size=(N_paths, N_steps))

    # increments of log S: dX = (r - 0.5 sigma^2) dt + sigma * sqrt(dt) * Z
    drift_inc = (r - 0.5 * sigma**2) * dt
    dX = drift_inc + sigma * sqrt_dt * Z

    # build X values: X_0 ... X_N_steps (log S), shape (N_paths, N_steps+1)
    X = np.concatenate([np.full((N_paths, 1), lnS0), np.cumsum(dX, axis=1) + lnS0], axis=1)

    # terminal S and intrinsic payoff
    X_T = X[:, -1]
    S_T = np.exp(X_T)
    intrinsic = np.maximum(S_T - K, 0.0)
    discount = math.exp(-r * T)

    # Brownian-bridge survival probabilities q_k on each interval
    Xk = X[:, :-1]   # X at left endpoint of interval
    Xk1 = X[:, 1:]   # X at right endpoint

    # intervals where both endpoints are above lnB (otherwise crossing occurred)
    both_above = (Xk > lnB) & (Xk1 > lnB)

    # A = (Xk - lnB)*(Xk1 - lnB)
    A = (Xk - lnB) * (Xk1 - lnB)

    # exponent = -2 * A / (sigma^2 dt)
    exponent = -2.0 * A / (sigma**2 * dt)

    # p_cross = exp(exponent)   (only meaningful when both_above is True)
    # q = 1 - p_cross  (survival prob for that interval)
    with np.errstate(over='ignore'):
        p_cross = np.exp(exponent)
    q = np.where(both_above, 1.0 - p_cross, 0.0)   # q==0 if any endpoint <= lnB

    # product of q over intervals = probability (under BB correction) that path stayed above barrier
    # compute in log space for stability
    with np.errstate(divide='ignore'):
        log_q = np.where(q > 0, np.log(q), -np.inf)
    log_prod_q = np.sum(log_q, axis=1)  # -inf if any q==0 -> product 0
    prod_q = np.exp(log_prod_q, where=np.isfinite(log_prod_q))
    prod_q = np.where(np.isfinite(log_prod_q), prod_q, 0.0)  # set -inf -> 0

    # adjusted (conditional) discounted payoff per path
    payoff_adj = discount * intrinsic * prod_q

    # --------------------------
    # Pathwise Delta (dV/dS0)
    # --------------------------
    # dS_T/dS0 = S_T / S0
    dST_dS0 = S_T / S0

    # derivatives of q wrt Xk and Xk1
    # q = 1 - exp(exponent) where exponent = -2A/(sigma^2 dt)
    # dq/dA = (2/(sigma^2 dt)) * exp(exponent)
    factor = np.exp(exponent)
    dq_dA = (2.0 / (sigma**2 * dt)) * factor

    dA_dXk = (Xk1 - lnB)
    dA_dXk1 = (Xk - lnB)

    dq_dXk = dq_dA * dA_dXk
    dq_dXk1 = dq_dA * dA_dXk1

    # zero derivatives where q is defined as 0 (both_above False)
    dq_dXk = np.where(both_above, dq_dXk, 0.0)
    dq_dXk1 = np.where(both_above, dq_dXk1, 0.0)

    # dX/dS0 = 1/S0 (log S depends additively on lnS0)
    dX_dS0_scalar = 1.0 / S0

    # derivative of product Q = prod q_i: dQ/dS0 = Q * sum_i ( (dq_i/dS0) / q_i )
    # where dq_i/dS0 = (dq_i/dXk + dq_i/dXk1) * (1/S0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(q > 0, (dq_dXk + dq_dXk1) / q, 0.0)
    dlogQ_dS0 = np.sum(ratio, axis=1) * dX_dS0_scalar
    dQ_dS0 = prod_q * dlogQ_dS0

    indicator = (intrinsic > 0).astype(float)
    delta_paths = discount * indicator * (dST_dS0 * prod_q + intrinsic * dQ_dS0)
    delta_est = np.mean(delta_paths)
    delta_se = np.std(delta_paths, ddof=1) / math.sqrt(N_paths)

    # --------------------------
    # Pathwise Vega (dV/dsigma)
    # --------------------------
    # derivative of X_k wrt sigma:
    # X_k = lnS0 + (r - 0.5 sigma^2) * k*dt + sigma*sqrt_dt * sum_{i<k} Z_i
    # => dX_k/dsigma = -sigma * k * dt + sqrt_dt * cumsum(Z) up to k-1
    cumsumZ = np.cumsum(Z, axis=1)  # shape (N_paths, N_steps)
    kidx = np.arange(1, N_steps + 1)
    term1 = -sigma * (kidx * dt)
    dX_dsigma_increments = term1.reshape(1, -1) + sqrt_dt * cumsumZ  # shape (N_paths, N_steps)
    dX_dsigma = np.concatenate([np.zeros((N_paths, 1)), dX_dsigma_increments], axis=1)  # include time 0

    # derivative of S_T wrt sigma: dS_T/dsigma = S_T * dX_T/dsigma
    dST_dsigma = S_T * dX_dsigma[:, -1]

    # derivative of q wrt sigma
    dA_dsigma = dX_dsigma[:, :-1] * (Xk1 - lnB) + (Xk - lnB) * dX_dsigma[:, 1:]
    # dexponent/dsigma = -2/dt * ( dA/dsigma / sigma^2 - 2A / sigma^3 )
    dexponent_dsigma = -2.0 / (dt) * (dA_dsigma / (sigma**2) - 2.0 * A / (sigma**3))

    dq_dsigma = - factor * dexponent_dsigma
    dq_dsigma = np.where(both_above, dq_dsigma, 0.0)

    # derivative of product Q wrt sigma: dQ/dsigma = Q * sum_j (dq_j/dsigma / q_j)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio_sigma = np.where(q > 0, dq_dsigma / q, 0.0)
    dlogQ_dsigma = np.sum(ratio_sigma, axis=1)
    dQ_dsigma = prod_q * dlogQ_dsigma

    vega_paths = discount * indicator * (dST_dsigma * prod_q + intrinsic * dQ_dsigma)
    vega_est = np.mean(vega_paths)
    vega_se = np.std(vega_paths, ddof=1) / math.sqrt(N_paths)

    price_est = np.mean(payoff_adj)
    price_se = np.std(payoff_adj, ddof=1) / math.sqrt(N_paths)

    return {
        'price': float(price_est),
        'price_se': float(price_se),
        'delta': float(delta_est),
        'delta_se': float(delta_se),
        'vega': float(vega_est),
        'vega_se': float(vega_se),
        'N_paths': N_paths,
        'N_steps': N_steps
    }

# ---------------------------
# If run as script, execute default case
# ---------------------------
if __name__ == "__main__":
    res = mc_barrier_pathwise(
        N_paths=100000,   # change to e.g. 20000 if you need faster runs
        N_steps=100,      # time discretization steps
        S0=100.0,
        K=110.0,
        B=90.0,
        r=0.05,
        sigma=0.5,
        T=1.0,
        seed=12345
    )

    print("Results (Monte Carlo with Brownian-bridge BB correction and pathwise sensitivities):")
    print(f"Price = {res['price']:.6f}  (SE = {res['price_se']:.6f})")
    print(f"Delta = {res['delta']:.6f}  (SE = {res['delta_se']:.6f})")
    print(f"Vega  = {res['vega']:.6f}  (SE = {res['vega_se']:.6f})")
    print(f"N_paths = {res['N_paths']}, N_steps = {res['N_steps']}")

Results (Monte Carlo with Brownian-bridge BB correction and pathwise sensitivities):
Price = 8.593296  (SE = 0.094257)
Delta = 0.862835  (SE = 0.013078)
Vega  = 4.219711  (SE = 0.358786)
N_paths = 100000, N_steps = 100
